In [30]:
pip install transformers datasets torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [31]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import torch

In [2]:
# Load training utterances and answers
with open("data/train_utt.txt", "r") as f:
    utterances = f.readlines()

with open("data/train_ans.txt", "r") as f:
    slots = f.readlines()

# Strip whitespace
utterances = [utt.strip() for utt in utterances]
slots = [slot.strip() for slot in slots]


Parsing slot annotations into dictionaries and pairing them with their respective utterances. This part was intended to help us understanding the parsing process for our utterances. 

Future implementation: for higher-level intent, we should do use_cases and do single canonical use cases. For example "school", "work". 

In [3]:
def parse_slots(slot):
    pairs = slot.split("|")[1:] 
    slot_dict = {}
    for pair in pairs:
        key, value = pair.split("=")
        slot_dict[key] = value
    return slot_dict

# Convert slot strings to dictionaries
slot_dicts = [parse_slots(slot) for slot in slots]

# Combine utterances with their parsed slot dictionaries
utterance_slot_pairs = list(zip(utterances, slot_dicts))

# Display the parsed dictionaries
for i, (utterance, slot_dict) in enumerate(utterance_slot_pairs[:10]):  # Limit to first 10
    print(f"Utterance {i+1}: {utterance}")
    print(f"{slot_dict}")
    print()

Utterance 1: Can you show me laptops from a specific brand, like Apple?
{'brand': 'Apple'}

Utterance 2: I need a budget-friendly laptop. What’s the cheapest one you have?
{'price': 'cheap'}

Utterance 3: I need a laptop with a powerful processor for heavy tasks. Any suggestions?
{'processor_tier': 'high'}

Utterance 4: Can you recommend a laptop with at least a Core i5 processor?
{'processor_tier': 'i5+'}

Utterance 5: I need a compact laptop with a smaller display. What are my options?
{'display_size': 'small'}

Utterance 6: What’s the most affordable laptop with a 15.6-inch display?
{'display_size': '15.6', 'price': 'cheap'}

Utterance 7: I am trying to find a laptop that has a core i9 processor and display size of 15.6, what are my options?
{'processor_tier': 'i9', 'display_size': '15.6'}

Utterance 8: I am looking specifically for a Samsung laptop.
{'brand': 'Samsung'}

Utterance 9: I want a laptop that has at least 16GB RAM and doesn’t cost more than $500. Any options?
{'ram_memo

In [4]:
from collections import defaultdict

# Initialize slot-specific contexts and answers
brand_context = []
brand_answers = []
price_context = []
price_answers = []
processor_context = []
processor_answers = []
display_context = []
display_answers = []

# Dictionary to store slots and their unique values
slot_dic = defaultdict(lambda: defaultdict(set))

# Load the files
with open("data/train_utt.txt") as f1, open("data/train_ans.txt") as f2:
    utterances = f1.readlines()
    for i, line in enumerate(f2.readlines()):
        a_line = line.strip()
        ans = a_line.split('|')
        # The intent (always 'find_laptop')
        intent = ans[0] 
        
        # Process slots in the answer
        for a in ans[1:]:
            if "!=" in a:  # Handle negated slots
                slot_name, slot_value = a.split("!=")
            else:  # Handle regular slots
                slot_name, slot_value = a.split("=")
            
            # Collect context and answers for specific slots
            if slot_name == "brand":
                brand_context.append(utterances[i].strip())
                brand_answers.append(slot_value)
            if slot_name == "price":
                price_context.append(utterances[i].strip())
                price_answers.append(slot_value)
            if slot_name == "processor_tier":
                processor_context.append(utterances[i].strip())
                processor_answers.append(slot_value)
            if slot_name == "display_size":
                display_context.append(utterances[i].strip())
                display_answers.append(slot_value)
            
            # Add slot values to the slot dictionary
            slot_dic[slot_name]["values"].add(slot_value)

# Display the slot dictionary
slot_dic

defaultdict(<function __main__.<lambda>()>,
            {'brand': defaultdict(set,
                         {'values': {'ASUS',
                           'Apple',
                           'Dell',
                           'HP',
                           'Microsoft',
                           'Samsung',
                           'lenovo'}}),
             'price': defaultdict(set,
                         {'values': {'500',
                           '700',
                           'cheap',
                           'expensive',
                           'mid-range'}}),
             'processor_tier': defaultdict(set,
                         {'values': {'core i7',
                           'decent',
                           'high',
                           'i5+',
                           'i7',
                           'i9',
                           'latest',
                           'm1',
                           'm2'}}),
             'display_size': defaultdict

In [42]:
brand_context[:3]

['Can you show me laptops from a specific brand, like Apple?',
 'I am looking specifically for a Samsung laptop.',
 'Can you recommend a laptop with a small display size but that is not an Apple brand?']

In [43]:
brand_answers[:3]

['Apple', 'Samsung', 'Apple']

Edge Case: Negation for specific 

In [10]:
import pandas as pd

# Load data
with open("data/train_utt.txt", "r") as f:
    utterances = [line.strip() for line in f.readlines()]

with open("data/train_ans.txt", "r") as f:
    annotations = [line.strip() for line in f.readlines()]

# Extract slots and create questions
questions, contexts, answers = [], [], []
for utt, ann in zip(utterances, annotations):
    context = utt
    slots = ann.split('|')[1:]  # Skip the intent
    for slot in slots:
        slot_name, slot_value = (slot.split('=') if '=' in slot else slot.split('!='))
        questions.append(f"What is the {slot_name.replace('_', ' ')}?")
        contexts.append(context)
        answers.append(slot_value)

# Save as DataFrame
data = pd.DataFrame({'context': contexts, 'question': questions, 'answer': answers})
data.head()


,context,question,answer
0,"Can you show me laptops from a specific brand,...",What is the brand?,Apple
1,I need a budget-friendly laptop. What’s the ch...,What is the price?,cheap
2,I need a laptop with a powerful processor for ...,What is the processor tier?,high
3,Can you recommend a laptop with at least a Cor...,What is the processor tier?,i5+
4,I need a compact laptop with a smaller display...,What is the display size?,small


In [11]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [16]:
def get_answer_span_tensor(question, context, answer):
    # Print inputs before tokenization
    print("Question:", question)
    print("Context:", context)
    print("Answer:", answer)

    # Tokenize inputs
    ques = tokenizer.tokenize(question)
    context = tokenizer.tokenize(context)
    answer = tokenizer.tokenize(answer)

    # Print tokenized inputs
    print("Tokenized Question:", ques)
    print("Tokenized Context:", context)
    print("Tokenized Answer:", answer)

    # Combine tokens with special tokens
    s = ["[CLS]"] + ques + ["[SEP]"] + context + ["[SEP]"]
    print("Combined Tokens (with special tokens):", s)

    # Initialize start and end
    start, end = 0, 0

    # Match tokens to find the answer span
    for id_a, token_a in enumerate(answer):
        for id_s, token_s in enumerate(s):
            if token_a == token_s:
                if answer == s[id_s:id_s + len(answer)]:
                    start = id_s
                    end = id_s + len(answer) - 1
                    break
                elif len(s) - id_s + 1 <= len(answer):
                    break

    # Print start and end positions
    print("Start:", start)
    print("End:", end)

    return torch.tensor([start, end])

# Example inputs
question = "What laptop has an M1 chip?"
context = "Apple laptops have an M1 chip."
answer = "Apple"

# Call function
tensor = get_answer_span_tensor(question, context, answer)
print("Answer Span Tensor:", tensor)


Question: What laptop has an M1 chip?
Context: Apple laptops have an M1 chip.
Answer: Apple
Tokenized Question: ['what', 'laptop', 'has', 'an', 'm1', 'chip', '?']
Tokenized Context: ['apple', 'laptop', '##s', 'have', 'an', 'm1', 'chip', '.']
Tokenized Answer: ['apple']
Combined Tokens (with special tokens): ['[CLS]', 'what', 'laptop', 'has', 'an', 'm1', 'chip', '?', '[SEP]', 'apple', 'laptop', '##s', 'have', 'an', 'm1', 'chip', '.', '[SEP]']
Start: 9
End: 9
Answer Span Tensor: tensor([9, 9])


In [17]:
test_question = "What laptop has an M1 chip?"
test_context = "Apple laptops have an M1 chip."
test_answer = "Apple"
bad_answer  = "HP"
span = get_answer_span_tensor(test_question,test_context,test_answer)
# print(span)
assert span.shape == (2,)
assert list(span) == [9,9]
span = get_answer_span_tensor(test_question,test_context,bad_answer)
assert list(span) == [0,0]
print('Success!')

Question: What laptop has an M1 chip?
Context: Apple laptops have an M1 chip.
Answer: Apple
Tokenized Question: ['what', 'laptop', 'has', 'an', 'm1', 'chip', '?']
Tokenized Context: ['apple', 'laptop', '##s', 'have', 'an', 'm1', 'chip', '.']
Tokenized Answer: ['apple']
Combined Tokens (with special tokens): ['[CLS]', 'what', 'laptop', 'has', 'an', 'm1', 'chip', '?', '[SEP]', 'apple', 'laptop', '##s', 'have', 'an', 'm1', 'chip', '.', '[SEP]']
Start: 9
End: 9
Question: What laptop has an M1 chip?
Context: Apple laptops have an M1 chip.
Answer: HP
Tokenized Question: ['what', 'laptop', 'has', 'an', 'm1', 'chip', '?']
Tokenized Context: ['apple', 'laptop', '##s', 'have', 'an', 'm1', 'chip', '.']
Tokenized Answer: ['hp']
Combined Tokens (with special tokens): ['[CLS]', 'what', 'laptop', 'has', 'an', 'm1', 'chip', '?', '[SEP]', 'apple', 'laptop', '##s', 'have', 'an', 'm1', 'chip', '.', '[SEP]']
Start: 0
End: 0
Success!


In [18]:
def convert_to_BERT_tensors(questions, contexts):
    '''takes a parallel list of question strings and answer strings'''
    #your code here
    result = tokenizer(questions,contexts,  return_tensors="pt", padding='max_length', truncation=True,  max_length=512)
    return result['input_ids'], result['attention_mask']

In [20]:
test_questions = ["Why?", "How?"]
test_contexts = ["I think it is because we can bluminate", "It was done"" ".join(["very"]*1000) + " well"]

ids, mask = convert_to_BERT_tensors(test_questions,test_contexts)
assert ids.shape == (2,512) # 512 because that's the max allowed
assert ids[0][3] == 102 # fourth token is separator
assert list(ids[0][-100:]) == [0]*100 # first row is mostly padding
assert list(ids[1][-100:]) != [0]*100 # second row is not
assert list(mask[0][-100:]) == [0]*100 # first row padding is masked
assert list(mask[1][-100:]) != [0]*100 # second row is not padding, no mask
print("Success!")

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Success!


In [21]:
a = [1,2]
b = [3,4]
a.extend(b)
a.extend([3]*4)
a

[1, 2, 3, 4, 3, 3, 3, 3]

In [22]:
#provided code
batch_size = 16

class QAdataset(Dataset):
    '''A dataset for housing QA data, including input_data, output_data, and padding mask'''
    def __init__(self, input_data, output_data,mask):
        self.input_data = input_data
        self.output_data = output_data
        self.mask = mask
        
    def __len__(self):
        return len(self.input_data)
    
    def __getitem__(self, index):
        target = self.output_data[index]
        data_val = self.input_data[index]
        mask = self.mask[index]
        return data_val,target,mask 

In [24]:
from torch.utils.data import DataLoader
import torch

# Initialize variables for training data
brand_answers = []
price_answers = []
processor_tier_answers = []
display_size_answers = []
ram_memory_answers = []
all_contexts = []

# Process the files
file_path_utt = "data/train_utt.txt"
file_path_ans = "data/train_ans.txt"

with open(file_path_utt) as f1, open(file_path_ans) as f2:
    utterances = f1.readlines()
    print(f"Number of utterances: {len(utterances)}")
    
    for i, line in enumerate(f2.readlines()):
        a_line = line.strip()
        slots = a_line.split('|')  # Split into intent and slot-value pairs
        intent = slots[0]

        # Ensure the intent matches `find_laptop`
        if intent != "find_laptop":
            continue

        # Append the context (utterance) for this example
        all_contexts.append(utterances[i].strip())

        # Initialize default values for slots
        brand = "XXNOANSWERXX"
        price = "XXNOANSWERXX"
        processor_tier = "XXNOANSWERXX"
        display_size = "XXNOANSWERXX"
        ram_memory = "XXNOANSWERXX"

        # Extract slot values
        for slot in slots[1:]:
            slot_key, slot_value = slot.split('=')
            if slot_key == 'brand':
                brand = slot_value
            elif slot_key == 'price':
                price = slot_value
            elif slot_key == 'processor_tier':
                processor_tier = slot_value
            elif slot_key == 'display_size':
                display_size = slot_value
            elif slot_key == 'ram_memory':
                ram_memory = slot_value

        # Append slot values to respective lists
        brand_answers.append(brand)
        price_answers.append(price)
        processor_tier_answers.append(processor_tier)
        display_size_answers.append(display_size)
        ram_memory_answers.append(ram_memory)

# Print lengths for debugging
print("Total contexts:", len(all_contexts))
print("Total brands:", len(brand_answers))
print("Total prices:", len(price_answers))
print("Total processor tiers:", len(processor_tier_answers))
print("Total display sizes:", len(display_size_answers))
print("Total RAMs:", len(ram_memory_answers))

# Generate questions for each slot
brand_questions = ["What brand is the user looking for?"] * len(brand_answers)
price_questions = ["What price range is the user looking for?"] * len(price_answers)
processor_tier_questions = ["What processor tier does the user prefer?"] * len(processor_tier_answers)
display_size_questions = ["What display size is the user looking for?"] * len(display_size_answers)
ram_memory_questions = ["How much RAM does the user need?"] * len(ram_memory_answers)

# Prepare DataLoader for each slot
def prepare_dataloader(questions, contexts, answers, batch_size):
    QA_input, masks = convert_to_BERT_tensors(questions, contexts)
    spans = [get_answer_span_tensor(questions[i], contexts[i], answers[i]) for i in range(len(questions))]
    return DataLoader(QAdataset(QA_input, spans, masks), batch_size=batch_size, num_workers=4, shuffle=False)

# DataLoaders for each slot
batch_size = 16

brand_train_dataloader = prepare_dataloader(brand_questions, all_contexts, brand_answers, batch_size)
price_train_dataloader = prepare_dataloader(price_questions, all_contexts, price_answers, batch_size)
processor_tier_train_dataloader = prepare_dataloader(processor_tier_questions, all_contexts, processor_tier_answers, batch_size)
display_size_train_dataloader = prepare_dataloader(display_size_questions, all_contexts, display_size_answers, batch_size)
ram_memory_train_dataloader = prepare_dataloader(ram_memory_questions, all_contexts, ram_memory_answers, batch_size)

print("DataLoaders created successfully.")

Number of utterances: 40
Total contexts: 40
Total brands: 40
Total prices: 40
Total processor tiers: 40
Total display sizes: 40
Total RAMs: 40
Question: What brand is the user looking for?
Context: Can you show me laptops from a specific brand, like Apple?
Answer: Apple
Tokenized Question: ['what', 'brand', 'is', 'the', 'user', 'looking', 'for', '?']
Tokenized Context: ['can', 'you', 'show', 'me', 'laptop', '##s', 'from', 'a', 'specific', 'brand', ',', 'like', 'apple', '?']
Tokenized Answer: ['apple']
Combined Tokens (with special tokens): ['[CLS]', 'what', 'brand', 'is', 'the', 'user', 'looking', 'for', '?', '[SEP]', 'can', 'you', 'show', 'me', 'laptop', '##s', 'from', 'a', 'specific', 'brand', ',', 'like', 'apple', '?', '[SEP]']
Start: 22
End: 22
Question: What brand is the user looking for?
Context: I need a budget-friendly laptop. What’s the cheapest one you have?
Answer: XXNOANSWERXX
Tokenized Question: ['what', 'brand', 'is', 'the', 'user', 'looking', 'for', '?']
Tokenized Contex

Combined Tokens (with special tokens): ['[CLS]', 'how', 'much', 'ram', 'does', 'the', 'user', 'need', '?', '[SEP]', 'i', '’', 'm', 'a', 'cs', 'student', 'and', 'i', 'need', 'a', 'laptop', 'that', 'can', 'run', 'linux', '[SEP]']
Start: 0
End: 0

Success! Results should be [0,0] because In this case, the answer is "XXNOANSWERXX" because no relevant ram_memory information is present in the context. The model is expected to predict a span [0, 0] when no valid answer exists.